# Iridium-50m on Kaggle

The model is untrained until this finishes. Expect several sessions, and
run each one with **Save Version -> Save & Run All**: a session that is not
saved as a version loses its output.

| Session | Accelerator | What it does | About |
|---|---|---|---|
| 1 | None (CPU, spends no GPU quota) | Downloads and tokenizes the data | up to 12 h |
| 2, 3, ... | **GPU T4 x2**, with internet on | Trains on both GPUs at once, stops itself before the 12 h limit | ~11 h each |

**Between sessions:** Add Input -> Notebook Output -> this notebook's latest
version with output. Attach the data version and the newest training
version. The notebook finds them itself.

Pick **GPU T4 x2**, not P100: training runs one process per GPU and
averages their gradients, so two T4s do each step in about half the time.
The hours are estimates. Nothing has measured this model's speed yet, so
read the `tok/s` figure on the log lines in the first training session. Sessions
can be repeated until the last cell reports the run finished.

## 1. Source (private repo)

Needs a Kaggle secret named `GITHUB_TOKEN`; it is never printed or stored.

In [ ]:
# This repository is private until Iridium 1.0 ships. This cell never
# hard-codes or prints a token: it reads one from the host's own secret
# store (or an env var on a plain Jupyter host) and hands it to git only
# through an environment-scoped header, so it never lands in .git/config.
import os, sys, subprocess
from pathlib import Path
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    token = None

REPO_OWNER = 'sporadicstudiosind-cloud'
REPO_NAME = 'test'
RELEASE_BRANCH = 'claude/gallant-faraday-lhycva'
ROOT = Path.cwd()

# The token travels in an environment-scoped git config header: never in
# the URL (git would store it in .git/config) and never in the argv (a
# failed subprocess prints its argv).
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
if token:
    import base64
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update(GIT_CONFIG_COUNT='1',
               GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {basic}')
    del basic

if (ROOT / 'iridium' / 'presets.py').is_file():
    print('Already inside a checkout of the repository:', ROOT)
else:
    ROOT = Path.cwd() / REPO_NAME
    if ROOT.is_dir():
        # A checkout left over from earlier in this session would
        # otherwise silently keep running whatever code it was cloned
        # with, including bugs fixed upstream since. Bring it up to date.
        # fetch + reset rather than pull --ff-only: a depth-1 clone has no
        # history for git to prove a fast-forward with.
        dirty = subprocess.run(['git', '-C', str(ROOT), 'status', '--porcelain',
                                '--untracked-files=no'],
                               capture_output=True, text=True).stdout.strip()
        if dirty:
            raise RuntimeError('The existing checkout at %s has local edits, so it was '
                               'not updated:\n%s\nDelete it (!rm -rf %s) and rerun this '
                               'cell for a fresh clone.' % (ROOT, dirty, ROOT))
        fetched = subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1',
                                  'origin', RELEASE_BRANCH],
                                 env=env, capture_output=True, text=True)
        if fetched.returncode != 0:
            raise RuntimeError('Could not update the existing checkout at %s (exit %d). '
                               'Delete it (!rm -rf %s) and rerun this cell.' %
                               (ROOT, fetched.returncode, ROOT))
        before = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                                capture_output=True, text=True).stdout.strip()
        subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', 'FETCH_HEAD'],
                       check=True, capture_output=True)
        after = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'],
                               capture_output=True, text=True).stdout.strip()
        print('Updated existing checkout at', ROOT)
        if before != after and 'iridium' in sys.modules:
            # New code on disk does not replace modules already imported:
            # the old ones keep running and the update never executes.
            raise RuntimeError('The checkout was just updated, but this kernel already '
                               'imported the old code. Restart the kernel, then run the '
                               'notebook from the top.')
    else:
        if not token:
            print('No GITHUB_TOKEN found. In Kaggle: Add-ons, Secrets, add GITHUB_TOKEN with a fine-grained PAT that can read this repository, then attach it to this notebook.')
            print('Trying an unauthenticated clone (works only if the repo is public)...')
        done = subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                               f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                               str(ROOT)], env=env, capture_output=True, text=True)
        if done.returncode != 0:
            raise RuntimeError('git clone failed (exit %d). Check the token can read '
                               'the repository and the branch exists.' % done.returncode)
        print('Cloned', REPO_OWNER + '/' + REPO_NAME, '@', RELEASE_BRANCH, 'to', ROOT)
del env
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()
print('source revision:', revision or 'unknown')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working directory:', Path.cwd())

## 2. Install and cost the run

In [ ]:
!pip install -q -e ".[data]"

In [ ]:
import time
SESSION_STARTED = time.monotonic()

import torch
from iridium.presets import get_preset, preset_table
from iridium.runtime.device import detect
from iridium.training.run_preset import dry_run

info = detect()
DEVICE = info.device
print(info.describe())
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print('GPUs:', names)
    if len(names) < 2:
        print('NOTE: one GPU. Settings -> Accelerator -> GPU T4 x2 trains about twice as fast.')

PRESET = 'iridium-50m'
preset = get_preset(PRESET)
print(preset_table())
result = dry_run(preset)
if not result['match']:
    raise RuntimeError(f"parameter count mismatch: {result}; do not train until this reconciles")

## 3. Find earlier sessions' work

In [ ]:
import re, shutil
from pathlib import Path

OUT_DIR = Path('/kaggle/working/iridium-runs')
RUN_DIR = OUT_DIR / PRESET
RUN_DIR.mkdir(parents=True, exist_ok=True)
INPUTS = Path('/kaggle/input')

def attached(pattern):
    return sorted(INPUTS.rglob(pattern)) if INPUTS.is_dir() else []

def round_of(path):
    found = re.search(r'-round(\d+)\.pt$', path.name)
    return int(found.group(1)) if found else -1

data_marks = attached(f'{PRESET}/prepared.json')
DATA_DIR = data_marks[0].parent.parent if data_marks else OUT_DIR / 'data'
print('data:', DATA_DIR, '(attached)' if data_marks else '(not attached)')

# Copy only the newest checkpoint: inputs are read-only, and disk is limited.
if not list(RUN_DIR.glob('*-round*.pt')):
    earlier = max(attached(f'{PRESET}-round*.pt'), key=round_of, default=None)
    if earlier is not None:
        print('resuming from', earlier)
        shutil.copy2(earlier, RUN_DIR / earlier.name)
    else:
        print('no earlier checkpoint attached: this run starts from step 0')

## 4. Prepare data

Skipped when shards are already here. Writes about 7.5 GB.

In [ ]:
from iridium.training.prepare import prepare

if not (DATA_DIR / PRESET / 'prepared.json').exists():
    if HAS_GPU:
        print('WARNING: preparing data with a GPU attached spends GPU quota. A CPU session is free.')
    prepare(preset, DATA_DIR)
    print('data written to', DATA_DIR)
if not HAS_GPU:
    print('Data is ready. Saved as this version\'s output: attach it and rerun with a GPU.')

## 5. Train

Resumes from the newest checkpoint and stops before the session limit.
`val_*_bpb` should fall over time. Lower is better.

In [ ]:
import subprocess, sys
from iridium.training.run_preset import find_latest

SESSION_LIMIT_HOURS = 11.0     # Kaggle kills a session at 12 h and may lose its output

checkpoint = None
if HAS_GPU:
    used = (time.monotonic() - SESSION_STARTED) / 3600
    # A separate process per GPU (`--gpus auto`), started from the shell so
    # this kernel never holds GPU memory the trainers need.
    subprocess.run([
        sys.executable, '-m', 'iridium', 'train', '--preset', PRESET,
        '--out', str(OUT_DIR), '--data', str(DATA_DIR), '--resume', 'auto',
        '--gpus', 'auto', '--keep-rounds', '1',
        '--time-budget-hours', f'{SESSION_LIMIT_HOURS - used:.2f}',
    ], check=True)
    checkpoint = find_latest(RUN_DIR)
    print('newest checkpoint:', checkpoint)

## 6. If it finished: export and grade

In [ ]:
import json, subprocess, sys

state = json.loads((RUN_DIR / 'preset.json').read_text()) if (RUN_DIR / 'preset.json').exists() else {}
if not state.get('trained'):
    print('Not finished yet: at step', state.get('completed_steps', 0), 'of', preset.steps,
          '- attach this version and run another GPU session.')
else:
    weights = OUT_DIR / f'{PRESET}-inference.pt'
    subprocess.run([sys.executable, '-m', 'iridium', 'export', '--checkpoint', str(checkpoint),
                    '--out', str(weights)], check=True)
    subprocess.run([sys.executable, '-m', 'iridium', 'battery', '--checkpoint', str(weights),
                    '--device', DEVICE, '--out', str(OUT_DIR / 'battery.json')], check=True)
    print('Download', weights, 'and battery.json from this version\'s output.')

## 7. On your own machine

```bash
python -m iridium chat --checkpoint iridium-50m-inference.pt
python -m iridium agent --checkpoint iridium-50m-inference.pt --workspace ./sandbox
python -m iridium agent --checkpoint iridium-50m-inference.pt --workspace ./sandbox --without web python
```

See `docs/iridium-50m.md` for the tools and what they can and cannot do.